# Notebook 09b — Análisis descriptivo temporal y sociodemográfico

## Objetivo

Completar el análisis descriptivo de la cartera de clientes de Selmark con el estudio del comportamiento temporal, el mix de canales operativos y el perfil sociodemográfico de la cartera nacional mediante el enriquecimiento MOSAIC. Este notebook constituye la segunda parte del Capítulo 4 de la memoria, complementaria al notebook 09a que abordó el análisis económico de la cartera.

## Estructura

1. Configuración y carga de datos
2. Análisis temporal del comportamiento de compra
3. Mix de canales y comportamiento de compra
4. Análisis sociodemográfico mediante MOSAIC
5. Síntesis y hallazgos para los siguientes capítulos

## Fuentes de datos

Los análisis combinan dos fuentes complementarias: la tabla `gold.cliente_360` para los agregados a nivel cliente (recencia, fechas extremas, porcentajes por temporada y perfiles MOSAIC) y la combinación de `silver.ventas_minoristas` con `silver.tiempo` para los análisis de granularidad temporal fina (evolución mensual real, impacto de eventos comerciales, cohortes). Esta doble aproximación permite caracterizar tanto el comportamiento individual de cada cliente como las dinámicas agregadas de la cartera en su conjunto.

## 1. Configuración y carga de datos

In [20]:
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# Rutas del proyecto
RUTA_PROYECTO = Path.home() / "OneDrive" / "Documentos" / "TFG_Selmark"
RUTA_DUCKDB = RUTA_PROYECTO / "duckdb" / "selmark.duckdb"
RUTA_FIGURAS = RUTA_PROYECTO / "output" / "figuras" / "09b"
RUTA_FIGURAS.mkdir(parents=True, exist_ok=True)

# Paleta corporativa coherente con 09a
COLOR_NACIONAL       = "#C8447F"
COLOR_INTERNACIONAL  = "#3E5C76"
COLOR_DESTACAR       = "#D4A017"
COLOR_POSITIVO       = "#5C9E76"
COLOR_NEGATIVO       = "#B85450"
COLOR_NEUTRO         = "#7D7D7D"
COLOR_FONDO          = "#FAFAFA"

PALETA_CATEGORICA = [
    "#C8447F", "#3E5C76", "#D4A017", "#5C9E76",
    "#B85450", "#7D7D7D", "#9B7BA6", "#E08E45"
]

# Plantilla plotly (misma configuración que en 09a)
TEMPLATE = go.layout.Template(
    layout=dict(
        font=dict(family="Arial, Helvetica, sans-serif", size=12, color="#1A1A1A"),
        title=dict(font=dict(size=16, color="#1A1A1A"), x=0.02, xanchor="left"),
        plot_bgcolor="white",
        paper_bgcolor="white",
        colorway=PALETA_CATEGORICA,
        xaxis=dict(showgrid=True, gridcolor="#EAEAEA", zeroline=False,
                   linecolor="#333333", linewidth=0.8, ticks="outside"),
        yaxis=dict(showgrid=True, gridcolor="#EAEAEA", zeroline=False,
                   linecolor="#333333", linewidth=0.8, ticks="outside"),
        margin=dict(l=60, r=40, t=70, b=60),
        legend=dict(bgcolor="rgba(255,255,255,0.95)", bordercolor="#CCCCCC", borderwidth=0.5),
    )
)
pio.templates["selmark"] = TEMPLATE
pio.templates.default = "selmark"

con = duckdb.connect(str(RUTA_DUCKDB), read_only=True)

def guardar_y_mostrar(fig, nombre, w=900, h=500):
    ruta_png = RUTA_FIGURAS / f"{nombre}.png"
    fig.write_image(str(ruta_png), width=w, height=h, scale=2)
    fig.show()
    print(f"   Figura guardada en: {ruta_png.name}")

def fmt_es(n, decimales=0):
    if pd.isna(n):
        return "—"
    if decimales == 0:
        return f"{int(n):,}".replace(",", ".")
    return f"{n:,.{decimales}f}".replace(",", "X").replace(".", ",").replace("X", ".")

print(f"Conexión establecida con: {RUTA_DUCKDB.name}")
print(f"Figuras se guardarán en: {RUTA_FIGURAS}")

Conexión establecida con: selmark.duckdb
Figuras se guardarán en: C:\Users\lopec\OneDrive\Documentos\TFG_Selmark\output\figuras\09b


In [21]:
# Carga de gold.cliente_360 y exclusión de cuentas técnicas para los análisis principales
df = con.execute("SELECT * FROM gold.cliente_360").fetchdf()

ids_cuentas_tecnicas = ["5728", "30942", "30957"]
df["es_cuenta_tecnica"] = df["id_cliente"].isin(ids_cuentas_tecnicas)
df_sin_tecnicas = df[~df["es_cuenta_tecnica"]].copy()

print(f"Tabla gold.cliente_360 cargada: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"Cartera sin cuentas técnicas:   {len(df_sin_tecnicas):,} filas")

Tabla gold.cliente_360 cargada: 3,492 filas × 57 columnas
Cartera sin cuentas técnicas:   3,489 filas


## 2. Análisis temporal del comportamiento de compra

Esta sección caracteriza la dinámica temporal de la cartera, mostrando la evolución mensual de la facturación, la estacionalidad propia del sector de la moda íntima (temporadas primavera-verano y otoño-invierno), el impacto de los principales eventos comerciales y la composición por cohortes según la fecha de primera operación de cada cliente.

In [22]:
# 2.1 Evolución mensual de la facturación retail 2022-2025
evol_mensual = con.execute("""
    SELECT
        DATE_TRUNC('month', vm.fecha_venta) AS mes,
        EXTRACT(YEAR FROM vm.fecha_venta) AS anio,
        SUM(vm.importe_total_con_descuento) AS facturacion_neta,
        COUNT(*) AS num_operaciones,
        COUNT(DISTINCT vm.id_cliente) AS clientes_activos
    FROM silver.ventas_minoristas vm
    WHERE vm.es_devolucion = FALSE
      AND vm.id_cliente NOT IN ('5728', '30942', '30957')
    GROUP BY mes, anio
    ORDER BY mes
""").fetchdf()

print(f"Periodos analizados: {len(evol_mensual)} meses ({evol_mensual['mes'].min().date()} a {evol_mensual['mes'].max().date()})")
print(f"Facturación retail acumulada (sin técnicas): {fmt_es(evol_mensual['facturacion_neta'].sum()/1e6, 2)} M €")

# Gráfico de líneas con un trazo por año
evol_mensual["anio"] = evol_mensual["anio"].astype(int)
evol_mensual["num_mes"] = evol_mensual["mes"].dt.month
evol_mensual["nombre_mes"] = evol_mensual["mes"].dt.month_name()

# Convertir nombres de mes al español
mapa_meses = {"January":"Ene","February":"Feb","March":"Mar","April":"Abr","May":"May","June":"Jun",
              "July":"Jul","August":"Ago","September":"Sep","October":"Oct","November":"Nov","December":"Dic"}
evol_mensual["nombre_mes_es"] = evol_mensual["nombre_mes"].map(mapa_meses)

fig = go.Figure()
colores_anios = {2022: COLOR_NACIONAL, 2023: COLOR_INTERNACIONAL, 2024: COLOR_DESTACAR, 2025: COLOR_POSITIVO}
for anio in sorted(evol_mensual["anio"].unique()):
    datos = evol_mensual[evol_mensual["anio"] == anio]
    fig.add_trace(go.Scatter(
        x=datos["nombre_mes_es"], y=datos["facturacion_neta"] / 1e6,
        mode="lines+markers", name=str(anio),
        line=dict(color=colores_anios.get(anio, COLOR_NEUTRO), width=2.5),
        marker=dict(size=8),
        hovertemplate="<b>%{x} " + str(anio) + "</b><br>Facturación: %{y:.2f} M €<extra></extra>"
    ))
fig.update_layout(
    title=dict(text="<b>Evolución mensual de la facturación retail 2022-2025</b><br><sup>Comparativa interanual (sin cuentas técnicas) · Facturación neta en millones de euros</sup>"),
    xaxis_title="Mes", yaxis_title="Facturación neta (M €)",
    height=520, legend=dict(title="Año", orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
guardar_y_mostrar(fig, "01_evolucion_mensual_facturacion")

Periodos analizados: 48 meses (2022-01-01 a 2025-12-01)
Facturación retail acumulada (sin técnicas): 67,10 M €


   Figura guardada en: 01_evolucion_mensual_facturacion.png


#### Nota metodológica sobre la cifra de facturación

La facturación retail acumulada que aparece en este notebook (67,10 millones de euros) es inferior a la cifra del notebook 09a (84,95 millones de euros) en exactamente 17,84 millones, importe que coincide con el aporte conjunto de las tres cuentas técnicas (REGO, HERREROS y EL CORTE INGLES). Esta diferencia es consecuencia directa de la decisión metodológica adoptada en el análisis 09a: el bloque temporal de este notebook trabaja sobre la cartera sin cuentas técnicas para obtener una lectura del comportamiento del mercado real, mientras que la cifra global del 09a incluye todas las cuentas para reflejar el volumen total documentado por el ERP.

In [23]:
# 2.2 Heatmap de estacionalidad (mes × año)
heatmap_data = evol_mensual.pivot_table(
    index="num_mes", columns="anio", values="facturacion_neta", aggfunc="sum"
) / 1e6
heatmap_data.index = ["Ene","Feb","Mar","Abr","May","Jun","Jul","Ago","Sep","Oct","Nov","Dic"]

fig = go.Figure(data=go.Heatmap(
    z=heatmap_data.values,
    x=[str(a) for a in heatmap_data.columns],
    y=heatmap_data.index,
    colorscale=[[0, "#FCEFEF"], [0.5, "#E89BB9"], [1, "#C8447F"]],
    text=heatmap_data.round(2).values,
    texttemplate="%{text} M€",
    textfont=dict(size=11, color="#1A1A1A"),
    hovertemplate="<b>%{y} %{x}</b><br>Facturación: %{z:.2f} M €<extra></extra>",
    colorbar=dict(title="M €", thickness=15),
))
fig.update_layout(
    title=dict(text="<b>Estacionalidad de la facturación retail · Mapa de calor mes × año</b><br><sup>Identificación de picos estacionales y meses fuertes recurrentes (sin cuentas técnicas)</sup>"),
    xaxis_title="Año", yaxis_title="Mes",
    height=550,
)
guardar_y_mostrar(fig, "02_heatmap_estacionalidad")

   Figura guardada en: 02_heatmap_estacionalidad.png


In [24]:
# 2.3 Impacto de eventos comerciales
eventos = con.execute("""
    SELECT 
        CASE
            WHEN t.es_black_friday = TRUE  THEN 'Black Friday'
            WHEN t.es_san_valentin = TRUE  THEN 'San Valentín'
            WHEN t.es_navidad = TRUE       THEN 'Navidad'
            WHEN t.es_rebajas = TRUE       THEN 'Rebajas'
            ELSE 'Días sin evento'
        END AS evento,
        COUNT(*) AS num_operaciones,
        SUM(vm.importe_total_con_descuento) AS facturacion,
        AVG(vm.importe_total_con_descuento) AS ticket_medio,
        COUNT(DISTINCT vm.fecha_venta) AS num_dias
    FROM silver.ventas_minoristas vm
    INNER JOIN silver.tiempo t ON vm.fecha_venta = t.fecha
    WHERE vm.es_devolucion = FALSE
      AND vm.id_cliente NOT IN ('5728', '30942', '30957')
    GROUP BY evento
    ORDER BY facturacion DESC
""").fetchdf()

eventos["facturacion_diaria_media"] = eventos["facturacion"] / eventos["num_dias"]

print("IMPACTO DE LOS EVENTOS COMERCIALES (cartera sin cuentas técnicas)")
print("=" * 75)
for _, row in eventos.iterrows():
    print(f"   {row['evento']:<18s}  {row['num_dias']:>4} días  {fmt_es(row['facturacion']/1e3, 0):>8s} K €  "
          f"Facturación/día: {fmt_es(row['facturacion_diaria_media']/1e3, 0):>6s} K €")

# Excluimos "Días sin evento" del gráfico para foco en eventos
eventos_solo = eventos[eventos["evento"] != "Días sin evento"].sort_values("facturacion_diaria_media", ascending=True)

fig = go.Figure(go.Bar(
    x=eventos_solo["facturacion_diaria_media"] / 1e3,
    y=eventos_solo["evento"],
    orientation="h",
    marker=dict(color=[COLOR_DESTACAR, COLOR_NACIONAL, COLOR_POSITIVO, COLOR_INTERNACIONAL][:len(eventos_solo)]),
    text=eventos_solo["facturacion_diaria_media"].apply(lambda v: f"{v/1e3:.0f} K €"),
    textposition="outside",
    hovertemplate="<b>%{y}</b><br>Facturación media/día: %{x:.1f} K €<extra></extra>",
))
# Línea de referencia: facturación media diaria en días sin evento
ref = eventos[eventos["evento"] == "Días sin evento"]["facturacion_diaria_media"].iloc[0] / 1e3
fig.add_vline(x=ref, line_dash="dash", line_color=COLOR_NEUTRO,
              annotation_text=f"Base sin eventos: {ref:.0f} K €/día",
              annotation_position="top")

fig.update_layout(
    title=dict(text="<b>Impacto de los eventos comerciales · Facturación media diaria</b><br><sup>Comparativa con la base de días sin evento (sin cuentas técnicas)</sup>"),
    xaxis_title="Facturación media diaria (miles de €)",
    yaxis_title="",
    height=480, showlegend=False,
)
guardar_y_mostrar(fig, "03_impacto_eventos_comerciales")

IMPACTO DE LOS EVENTOS COMERCIALES (cartera sin cuentas técnicas)
   Días sin evento     1078 días    49.158 K €  Facturación/día:     45 K €
   Rebajas              247 días    15.859 K €  Facturación/día:     64 K €
   Navidad              124 días     1.596 K €  Facturación/día:     12 K €
   San Valentín           4 días       426 K €  Facturación/día:    106 K €
   Black Friday           4 días        63 K €  Facturación/día:     15 K €


   Figura guardada en: 03_impacto_eventos_comerciales.png


#### Nota metodológica sobre el sesgo de truncamiento

El análisis de cohortes que se presenta a continuación utiliza como criterio el año de la primera operación retail documentada en la ventana temporal del proyecto (2022-2025). Esta aproximación presenta un sesgo de truncamiento conocido: los clientes que ya operaban con Selmark antes del 1 de enero de 2022 aparecen artificialmente clasificados como "cohorte 2022", cuando en realidad podrían ser clientes con varios años de antigüedad previa. Por tanto, la barra de la cohorte 2022 acumula tanto a los clientes genuinamente nuevos de ese año como al stock histórico preexistente.

Esta limitación es estructural y derivada del alcance temporal del proyecto. La interpretación correcta del gráfico debe centrarse en la comparativa entre las cohortes 2023, 2024 y 2025, que sí representan captaciones reales en cada año natural, dado que su fecha de primera operación cae dentro de la ventana de datos. La cohorte 2022 debe leerse como la base agregada del negocio en el momento de inicio de los datos del proyecto.

In [25]:
# 2.4 Análisis de cohortes por antigüedad
df_sin_tecnicas["anio_primera_op"] = pd.to_datetime(df_sin_tecnicas["fecha_primera_operacion"], errors="coerce").dt.year

cohortes = (df_sin_tecnicas[df_sin_tecnicas["anio_primera_op"].notna()]
            .groupby(["anio_primera_op", "tipo_mercado"])
            .agg(num_clientes=("id_cliente", "count"),
                 facturacion_combinada=("facturacion_b2b", lambda x: x.sum() + df_sin_tecnicas.loc[x.index, "facturacion_retail_neta_eur"].sum()))
            .reset_index())
cohortes["anio_primera_op"] = cohortes["anio_primera_op"].astype(int)

print("DISTRIBUCIÓN DE LA CARTERA POR AÑO DE PRIMERA OPERACIÓN")
print("=" * 75)
totales_anio = cohortes.groupby("anio_primera_op")["num_clientes"].sum()
for anio, n in totales_anio.items():
    print(f"   Año {anio}: {fmt_es(n):>6s} clientes con primera operación documentada")

fig = px.bar(
    cohortes, x="anio_primera_op", y="num_clientes",
    color="tipo_mercado",
    color_discrete_map={"NACIONAL": COLOR_NACIONAL, "INTERNACIONAL": COLOR_INTERNACIONAL},
    barmode="stack",
    text="num_clientes",
)
fig.update_traces(
    texttemplate="%{text:,}", textposition="inside",
    hovertemplate="<b>Año %{x}</b><br>%{fullData.name}: %{y:,} clientes<extra></extra>"
)
fig.update_layout(
    title=dict(text="<b>Cohortes de cartera por año de primera operación</b><br><sup>Distribución de clientes según el año en el que realizaron su primera operación retail</sup>"),
    xaxis_title="Año de primera operación", yaxis_title="Número de clientes",
    legend=dict(title="Mercado"), height=500,
)
guardar_y_mostrar(fig, "04_cohortes_antiguedad")

DISTRIBUCIÓN DE LA CARTERA POR AÑO DE PRIMERA OPERACIÓN
   Año 2022:  2.244 clientes con primera operación documentada
   Año 2023:    286 clientes con primera operación documentada
   Año 2024:    262 clientes con primera operación documentada
   Año 2025:    238 clientes con primera operación documentada


   Figura guardada en: 04_cohortes_antiguedad.png


### Conclusiones del análisis temporal

La caracterización temporal de la facturación retail revela un patrón estacional muy marcado y estable a lo largo del cuatrienio. Los picos sistemáticos de actividad coinciden con la lógica del sector de la moda íntima: campañas de rebajas en enero y julio, refuerzo navideño en diciembre y arranque de las temporadas primavera-verano y otoño-invierno en marzo y septiembre respectivamente. La estabilidad interanual de estos picos confirma la madurez del negocio y permite construir modelos predictivos posteriores con expectativa razonable de estacionariedad.

#### Hallazgo principal del análisis de eventos comerciales

El análisis cuantitativo del impacto diario de los eventos comerciales revela uno de los hallazgos más relevantes del Capítulo 4: **San Valentín constituye el evento de mayor intensidad para Selmark**, con una facturación media diaria de aproximadamente 106 mil euros, más del doble de la facturación de un día ordinario (45 mil euros). Este hallazgo, especialmente coherente con la naturaleza del producto (moda íntima femenina), supera con notable diferencia incluso a las rebajas (64 mil euros por día) y resulta particularmente revelador frente al débil impacto del Black Friday (15 mil euros por día, por debajo de la base ordinaria).

El comportamiento contrario en Black Friday confirma una característica sectorial frecuentemente discutida: la lencería tradicional no participa del fenómeno comercial del Black Friday en la misma medida que las categorías tecnológicas o de gran consumo. Este contraste tiene implicaciones estratégicas directas: la planificación comercial de Selmark debería concentrar recursos en el periodo previo a San Valentín y en las campañas de rebajas, mientras que la inversión en acciones específicas para Black Friday presenta un retorno previsiblemente bajo.

La interpretación de la facturación diaria de Navidad debe leerse con cautela, dado que la columna `es_navidad` de la tabla `silver.tiempo` cubre un periodo muy amplio (124 días en el cuatrienio, aproximadamente todo el mes de diciembre y parte de enero) en lugar de los días estrictamente navideños. La cifra de 12 mil euros por día corresponde al promedio de este periodo extendido, por lo que no resulta directamente comparable con eventos de marca diaria como San Valentín o Black Friday.

#### Cohortes de captación

El análisis de cohortes por año de primera operación informa sobre la dinámica de captación de la cartera, con las consideraciones metodológicas señaladas en la nota previa. Las cohortes 2023, 2024 y 2025 muestran captaciones netas anuales del orden de 200 a 300 clientes nuevos, un ritmo de incorporación que se mantendrá como referencia base en los análisis del Capítulo 7 (churn) para evaluar el balance entre captación y abandono.

## 3. Mix de canales y comportamiento de compra

In [26]:
# 3.1 Distribución por canal principal
canal_dist = (df_sin_tecnicas[df_sin_tecnicas["canal_principal"].notna()]
              .groupby("canal_principal")
              .agg(num_clientes=("id_cliente", "count"),
                   facturacion_retail=("facturacion_retail_neta_eur", "sum"))
              .reset_index()
              .sort_values("num_clientes", ascending=False))
canal_dist["pct_clientes"] = (100 * canal_dist["num_clientes"] / canal_dist["num_clientes"].sum()).round(2)

print("DISTRIBUCIÓN DE LA CARTERA POR CANAL PRINCIPAL DE COMPRA")
print("=" * 75)
for _, row in canal_dist.iterrows():
    print(f"   {str(row['canal_principal']):<25s}  {fmt_es(row['num_clientes']):>6s} cli ({row['pct_clientes']:>5.2f} %)  Fact retail: {fmt_es(row['facturacion_retail']/1e6, 2):>6s} M €")

fig = px.bar(
    canal_dist.head(10).sort_values("num_clientes"),
    x="num_clientes", y="canal_principal",
    orientation="h",
    color="facturacion_retail",
    color_continuous_scale=[[0, "#EFD9E5"], [1, COLOR_NACIONAL]],
    text="num_clientes",
)
fig.update_traces(
    texttemplate="%{text:,}", textposition="outside",
    hovertemplate="<b>%{y}</b><br>Clientes: %{x:,}<br>Facturación retail: %{marker.color:,.0f} €<extra></extra>"
)
fig.update_layout(
    title=dict(text="<b>Distribución de la cartera por canal principal de compra</b><br><sup>Número de clientes por canal y facturación retail asociada</sup>"),
    xaxis_title="Número de clientes", yaxis_title="",
    coloraxis_colorbar=dict(title="Facturación<br>retail (€)"),
    height=500,
)
guardar_y_mostrar(fig, "05_distribucion_canal_principal")

DISTRIBUCIÓN DE LA CARTERA POR CANAL PRINCIPAL DE COMPRA
   Temporada                   2.138 cli (70.08 %)  Fact retail:  56,82 M €
   Repetición                    734 cli (24.06 %)  Fact retail:   8,50 M €
   B2B                            67 cli ( 2.20 %)  Fact retail:   0,72 M €
   Muestras                       57 cli ( 1.87 %)  Fact retail:   0,15 M €
   Devolución                     34 cli ( 1.11 %)  Fact retail:   0,34 M €
   Dev. Muestras                  12 cli ( 0.39 %)  Fact retail:   0,06 M €
   ECI                             4 cli ( 0.13 %)  Fact retail:   0,07 M €
   PERSONALES                      3 cli ( 0.10 %)  Fact retail:   0,00 M €
   Depósito                        1 cli ( 0.03 %)  Fact retail:   0,33 M €
   Dev. Depósito                   1 cli ( 0.03 %)  Fact retail:   0,12 M €


   Figura guardada en: 05_distribucion_canal_principal.png


In [27]:
# 3.2 Estacionalidad por tipo de pedido (Temporada vs Repetición)
tipo_pedido_dist = (df_sin_tecnicas[df_sin_tecnicas["num_operaciones"] > 0]
                    .assign(pct_temp=lambda d: d["pct_pedidos_temporada"].fillna(0),
                            pct_rep=lambda d: d["pct_pedidos_repeticion"].fillna(0)))

resumen_temp_rep = pd.DataFrame({
    "Tipo de pedido": ["Pedidos de Temporada", "Pedidos de Repetición"],
    "pct_medio": [tipo_pedido_dist["pct_temp"].mean(), tipo_pedido_dist["pct_rep"].mean()],
    "pct_mediano": [tipo_pedido_dist["pct_temp"].median(), tipo_pedido_dist["pct_rep"].median()],
})

print("PORCENTAJE MEDIO DE PEDIDOS POR TIPO (clientes con actividad retail)")
print("=" * 70)
for _, row in resumen_temp_rep.iterrows():
    print(f"   {row['Tipo de pedido']:<25s}  Media: {row['pct_medio']:>5.2f} %  Mediana: {row['pct_mediano']:>5.2f} %")

# Boxplot comparativo
fig = go.Figure()
fig.add_trace(go.Box(
    y=tipo_pedido_dist["pct_temp"], name="Pedidos de Temporada",
    marker_color=COLOR_NACIONAL, boxmean=True
))
fig.add_trace(go.Box(
    y=tipo_pedido_dist["pct_rep"], name="Pedidos de Repetición",
    marker_color=COLOR_INTERNACIONAL, boxmean=True
))
fig.update_layout(
    title=dict(text="<b>Distribución del porcentaje de pedidos por tipo · Por cliente</b><br><sup>Comparativa entre pedidos de temporada y de repetición (sin cuentas técnicas)</sup>"),
    yaxis_title="% de pedidos del cliente",
    height=500, showlegend=False,
)
guardar_y_mostrar(fig, "06_temporada_vs_repeticion")

PORCENTAJE MEDIO DE PEDIDOS POR TIPO (clientes con actividad retail)
   Pedidos de Temporada       Media: 60.07 %  Mediana: 70.06 %
   Pedidos de Repetición      Media: 31.45 %  Mediana: 20.47 %


   Figura guardada en: 06_temporada_vs_repeticion.png


In [28]:
# 3.3 Multicanal vs monocanal (cuántos canales distintos usa cada cliente)
# Esta métrica la calculamos sobre silver.ventas_minoristas
diversidad = con.execute("""
    SELECT
        id_cliente,
        COUNT(DISTINCT id_tipo_pedido) AS num_canales_distintos
    FROM silver.ventas_minoristas
    WHERE es_devolucion = FALSE
      AND id_cliente NOT IN ('5728', '30942', '30957')
    GROUP BY id_cliente
""").fetchdf()

diversidad["categoria"] = pd.cut(
    diversidad["num_canales_distintos"],
    bins=[0, 1, 2, 3, 100],
    labels=["1 canal (monocanal)", "2 canales", "3 canales", "4+ canales"],
    include_lowest=True
)
distrib_div = diversidad.groupby("categoria", observed=True).size().reset_index(name="num_clientes")
distrib_div["pct"] = (100 * distrib_div["num_clientes"] / distrib_div["num_clientes"].sum()).round(2)

print("DIVERSIDAD DE CANALES POR CLIENTE (sin cuentas técnicas)")
print("=" * 65)
for _, row in distrib_div.iterrows():
    print(f"   {str(row['categoria']):<25s}  {fmt_es(row['num_clientes']):>6s} cli ({row['pct']:>5.2f} %)")

fig = px.bar(
    distrib_div, x="categoria", y="num_clientes",
    color="categoria",
    color_discrete_sequence=[COLOR_NEGATIVO, COLOR_NACIONAL, COLOR_DESTACAR, COLOR_POSITIVO],
    text="num_clientes",
)
fig.update_traces(
    texttemplate="%{text:,}", textposition="outside",
    hovertemplate="<b>%{x}</b><br>Clientes: %{y:,}<extra></extra>"
)
fig.update_layout(
    title=dict(text="<b>Diversidad de canales por cliente</b><br><sup>Distribución según el número de canales distintos en los que opera cada cliente</sup>"),
    xaxis_title="Categoría de diversidad",
    yaxis_title="Número de clientes",
    height=480, showlegend=False,
)
guardar_y_mostrar(fig, "07_multicanal_vs_monocanal")

DIVERSIDAD DE CANALES POR CLIENTE (sin cuentas técnicas)
   1 canal (monocanal)           792 cli (26.13 %)
   2 canales                   1.769 cli (58.36 %)
   3 canales                     454 cli (14.98 %)
   4+ canales                     16 cli ( 0.53 %)


   Figura guardada en: 07_multicanal_vs_monocanal.png


### Conclusiones del análisis del mix de canales

La caracterización del comportamiento de canales evidencia que el negocio retail de Selmark presenta una diversificación operativa significativa. La distribución por canal principal muestra una concentración esperable en los canales mayoritarios, pero también la presencia de canales especializados con perfiles de cliente diferenciados.

El análisis del porcentaje de pedidos por tipo (temporada frente a repetición) constituye un indicador clave del comportamiento de planificación de cada cliente. Los pedidos de temporada reflejan compromiso con las campañas de lanzamiento de colección, característico de clientes profesionalizados, mientras que los pedidos de repetición reflejan reposición a demanda. La distribución observada sugiere la coexistencia de perfiles de cliente con comportamientos de planificación claramente distintos.

La diversidad de canales por cliente añade una dimensión adicional: los clientes que operan en múltiples canales presentan un perfil más maduro y previsiblemente más vinculado a la marca, mientras que los clientes monocanal pueden representar relaciones menos consolidadas o necesidades específicas. Esta distinción será incorporada como variable explicativa en el clustering del Capítulo 5.

## 4. Análisis sociodemográfico mediante MOSAIC

El enriquecimiento sociodemográfico con la base de Experian (tabla `silver.mosaic`) se aplica exclusivamente sobre la cartera nacional con una cobertura del 96,82 % (2.038 de 2.105 clientes). Esta sección caracteriza la cartera según los perfiles MOSAIC asignados y cruza la información sociodemográfica con la facturación documentada.

In [29]:
# 4.1 Distribución por grupo MOSAIC (cartera nacional con MOSAIC asignado)
df_nacional_mosaic = df_sin_tecnicas[
    (df_sin_tecnicas["tipo_mercado"] == "NACIONAL") & 
    (df_sin_tecnicas["mosaic_grupo"].notna())
].copy()

grupos_mosaic = (df_nacional_mosaic.groupby("mosaic_grupo")
                 .agg(num_clientes=("id_cliente", "count"),
                      facturacion_b2b=("facturacion_b2b", "sum"),
                      facturacion_retail=("facturacion_retail_neta_eur", "sum"))
                 .reset_index()
                 .sort_values("num_clientes", ascending=False))
grupos_mosaic["facturacion_total"] = grupos_mosaic["facturacion_b2b"] + grupos_mosaic["facturacion_retail"]
grupos_mosaic["pct_clientes"] = (100 * grupos_mosaic["num_clientes"] / grupos_mosaic["num_clientes"].sum()).round(2)

print(f"DISTRIBUCIÓN DE LA CARTERA NACIONAL POR GRUPO MOSAIC ({len(df_nacional_mosaic):,} clientes)")
print("=" * 90)
for _, row in grupos_mosaic.head(15).iterrows():
    print(f"   {str(row['mosaic_grupo'])[:40]:<40s}  {fmt_es(row['num_clientes']):>5s} cli ({row['pct_clientes']:>5.2f} %)  Facturación total: {fmt_es(row['facturacion_total']/1e6, 2):>6s} M €")

fig = px.bar(
    grupos_mosaic.head(15).sort_values("num_clientes"),
    x="num_clientes", y="mosaic_grupo",
    orientation="h",
    color="facturacion_total",
    color_continuous_scale=[[0, "#EFD9E5"], [1, COLOR_NACIONAL]],
    text="num_clientes",
)
fig.update_traces(
    texttemplate="%{text:,}", textposition="outside",
    hovertemplate="<b>%{y}</b><br>Clientes: %{x:,}<br>Facturación total: %{marker.color:,.0f} €<extra></extra>"
)
fig.update_layout(
    title=dict(text="<b>Top 15 grupos MOSAIC de la cartera nacional</b><br><sup>Número de clientes por grupo sociodemográfico y facturación total asociada</sup>"),
    xaxis_title="Número de clientes",
    yaxis_title="",
    coloraxis_colorbar=dict(title="Facturación<br>total (€)"),
    height=600,
)
guardar_y_mostrar(fig, "08_grupos_mosaic", h=600)

DISTRIBUCIÓN DE LA CARTERA NACIONAL POR GRUPO MOSAIC (2,036 clientes)
   H                                           505 cli (24.80 %)  Facturación total:  17,01 M €
   C                                           314 cli (15.42 %)  Facturación total:  10,62 M €
   B                                           231 cli (11.35 %)  Facturación total:   7,56 M €
   I                                           204 cli (10.02 %)  Facturación total:   7,95 M €
   A                                           181 cli ( 8.89 %)  Facturación total:   6,65 M €
   G                                           178 cli ( 8.74 %)  Facturación total:   6,70 M €
   E                                           135 cli ( 6.63 %)  Facturación total:   4,58 M €
   D                                           115 cli ( 5.65 %)  Facturación total:   4,19 M €
   J                                            84 cli ( 4.13 %)  Facturación total:   2,62 M €
   F                                            47 cli ( 2.31 %)  

   Figura guardada en: 08_grupos_mosaic.png


In [30]:
# 4.2 Distribución por perfil comercial (los 5 flags de perfil)
perfiles = pd.DataFrame({
    "Perfil comercial": [
        "Premium", "Familiar joven", "Turístico", "Rural", "Sensible al precio"
    ],
    "Columna": [
        "perfil_premium", "perfil_familiar_joven", "perfil_turistico",
        "perfil_rural", "perfil_precio_sensible"
    ],
})
perfiles["num_clientes"] = perfiles["Columna"].apply(
    lambda c: int(df_nacional_mosaic[c].fillna(False).sum())
)
perfiles["pct_sobre_mosaic"] = (100 * perfiles["num_clientes"] / len(df_nacional_mosaic)).round(2)
perfiles["facturacion_total"] = perfiles["Columna"].apply(
    lambda c: df_nacional_mosaic.loc[df_nacional_mosaic[c].fillna(False),
                                        ["facturacion_b2b", "facturacion_retail_neta_eur"]].sum().sum()
)
perfiles["ticket_medio_cliente"] = perfiles["facturacion_total"] / perfiles["num_clientes"].replace(0, np.nan)

print(f"DISTRIBUCIÓN POR PERFIL COMERCIAL (sobre {len(df_nacional_mosaic):,} clientes con MOSAIC)")
print("=" * 80)
for _, row in perfiles.iterrows():
    print(f"   {row['Perfil comercial']:<20s}  {fmt_es(row['num_clientes']):>5s} cli ({row['pct_sobre_mosaic']:>5.2f} %)  "
          f"Fact total: {fmt_es(row['facturacion_total']/1e6, 2):>5s} M €  Fact/cliente: {fmt_es(row['ticket_medio_cliente'], 0):>7s} €")

# Visualización doble: número de clientes y ticket medio por perfil
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Número de clientes por perfil", "Facturación media por cliente y perfil"),
    horizontal_spacing=0.15,
)
fig.add_trace(go.Bar(
    x=perfiles["Perfil comercial"], y=perfiles["num_clientes"],
    marker_color=PALETA_CATEGORICA[:5],
    text=perfiles["num_clientes"].apply(lambda v: f"{v:,}".replace(",", ".")),
    textposition="outside", showlegend=False,
), row=1, col=1)
fig.add_trace(go.Bar(
    x=perfiles["Perfil comercial"], y=perfiles["ticket_medio_cliente"],
    marker_color=PALETA_CATEGORICA[:5],
    text=perfiles["ticket_medio_cliente"].apply(lambda v: f"{int(v):,} €".replace(",", ".") if pd.notna(v) else "—"),
    textposition="outside", showlegend=False,
), row=1, col=2)
fig.update_layout(
    title=dict(text="<b>Caracterización de la cartera por perfil comercial MOSAIC</b><br><sup>Comparativa entre número de clientes y facturación media por perfil</sup>"),
    height=520, showlegend=False,
)
fig.update_xaxes(tickangle=-25)
guardar_y_mostrar(fig, "09_perfiles_comerciales", h=520)

DISTRIBUCIÓN POR PERFIL COMERCIAL (sobre 2,036 clientes con MOSAIC)
   Premium                 495 cli (24.31 %)  Fact total: 17,27 M €  Fact/cliente:  34.893 €
   Familiar joven          250 cli (12.28 %)  Fact total:  8,77 M €  Fact/cliente:  35.075 €
   Turístico                47 cli ( 2.31 %)  Fact total:  2,06 M €  Fact/cliente:  43.747 €
   Rural                   124 cli ( 6.09 %)  Fact total:  3,77 M €  Fact/cliente:  30.374 €
   Sensible al precio      749 cli (36.79 %)  Fact total: 26,09 M €  Fact/cliente:  34.837 €


   Figura guardada en: 09_perfiles_comerciales.png


In [31]:
# 4.3 Renta media de los códigos postales con clientes
renta_data = df_nacional_mosaic[df_nacional_mosaic["renta_media"].notna()].copy()

if len(renta_data) > 0:
    print(f"ANÁLISIS DE RENTA MEDIA EN CP DE LA CARTERA NACIONAL ({len(renta_data):,} clientes)")
    print("=" * 75)
    print(f"   Renta media de la cartera:    {fmt_es(renta_data['renta_media'].mean(), 0):>10s} €")
    print(f"   Renta mediana de la cartera:  {fmt_es(renta_data['renta_media'].median(), 0):>10s} €")
    print(f"   Renta mínima:                 {fmt_es(renta_data['renta_media'].min(), 0):>10s} €")
    print(f"   Renta máxima:                 {fmt_es(renta_data['renta_media'].max(), 0):>10s} €")

    # Histograma
    fig = px.histogram(
        renta_data, x="renta_media", nbins=40,
        color_discrete_sequence=[COLOR_NACIONAL],
        opacity=0.85,
    )
    fig.add_vline(x=renta_data["renta_media"].mean(), line_dash="dash", line_color=COLOR_DESTACAR,
                  annotation_text=f"Media: {renta_data['renta_media'].mean():,.0f} €", annotation_position="top right")
    fig.add_vline(x=renta_data["renta_media"].median(), line_dash="dot", line_color=COLOR_POSITIVO,
                  annotation_text=f"Mediana: {renta_data['renta_media'].median():,.0f} €", annotation_position="top left")
    fig.update_layout(
        title=dict(text="<b>Distribución de la renta media de los códigos postales con cartera nacional</b><br><sup>Estimación Experian MOSAIC · renta media anual de los hogares del CP</sup>"),
        xaxis_title="Renta media estimada del CP (€)",
        yaxis_title="Número de clientes",
        height=500, showlegend=False,
    )
    guardar_y_mostrar(fig, "10_renta_media_cp")
else:
    print("Sin datos de renta media disponibles en la cartera con MOSAIC.")

ANÁLISIS DE RENTA MEDIA EN CP DE LA CARTERA NACIONAL (2,032 clientes)
   Renta media de la cartera:        25.891 €
   Renta mediana de la cartera:      26.289 €
   Renta mínima:                      4.499 €
   Renta máxima:                     35.903 €


   Figura guardada en: 10_renta_media_cp.png


### 4.4 Relación entre renta media del CP y facturación del cliente

Una pregunta natural en el análisis sociodemográfico es si la renta media estimada del código postal del cliente correlaciona con su facturación efectiva en Selmark. El siguiente gráfico de dispersión cruza ambas variables sobre la cartera nacional con MOSAIC y facturación combinada (B2B más retail) documentada.

In [32]:
# 4.4 Relación entre renta media del CP y facturación combinada del cliente
scatter_data = df_nacional_mosaic[df_nacional_mosaic["renta_media"].notna()].copy()
scatter_data["facturacion_combinada"] = (
    scatter_data["facturacion_b2b"].fillna(0) + 
    scatter_data["facturacion_retail_neta_eur"].fillna(0)
)
scatter_data = scatter_data[scatter_data["facturacion_combinada"] > 0]

# Calcular correlación de Spearman (robusta frente a outliers)
from scipy.stats import spearmanr
corr_s, p_s = spearmanr(scatter_data["renta_media"], scatter_data["facturacion_combinada"])

print("ANÁLISIS DE CORRELACIÓN: renta media del CP × facturación del cliente")
print("=" * 75)
print(f"   Clientes analizados:           {len(scatter_data):>5,}")
print(f"   Correlación de Spearman:       {corr_s:>+.4f}")
print(f"   p-valor:                        {p_s:>.4e}")
print(f"   Interpretación: ", end="")
if abs(corr_s) < 0.1:
    print("correlación inexistente o muy débil")
elif abs(corr_s) < 0.3:
    print("correlación débil")
elif abs(corr_s) < 0.5:
    print("correlación moderada")
else:
    print("correlación fuerte")

# Scatter con escala log en facturación para gestionar la asimetría
fig = px.scatter(
    scatter_data, x="renta_media", y="facturacion_combinada",
    color="mosaic_grupo",
    color_discrete_sequence=PALETA_CATEGORICA,
    hover_data={"nombre_cliente": True, "localidad_cliente": True,
                 "mosaic_grupo": True, "renta_media": ":,.0f",
                 "facturacion_combinada": ":,.0f"},
    opacity=0.65,
    log_y=True,
)
fig.update_traces(marker=dict(size=7, line=dict(width=0.5, color="white")))
fig.update_layout(
    title=dict(text=f"<b>Relación entre renta media del CP y facturación del cliente</b><br><sup>Cartera nacional con MOSAIC · escala logarítmica · Spearman ρ = {corr_s:+.3f}</sup>"),
    xaxis_title="Renta media estimada del CP (€)",
    yaxis_title="Facturación combinada del cliente (€) · escala log",
    height=600,
    legend=dict(title="Grupo MOSAIC", orientation="v"),
)
guardar_y_mostrar(fig, "11_scatter_renta_facturacion", w=950, h=600)

ANÁLISIS DE CORRELACIÓN: renta media del CP × facturación del cliente
   Clientes analizados:           1,805
   Correlación de Spearman:       +0.0043
   p-valor:                        8.5663e-01
   Interpretación: correlación inexistente o muy débil


   Figura guardada en: 11_scatter_renta_facturacion.png


#### Conclusiones de la relación renta-facturación

El análisis de correlación entre la renta media del código postal del cliente y su facturación combinada en Selmark arroja un resultado especialmente revelador: la correlación de Spearman se sitúa en +0,0043 con un p-valor de 0,8566, lo que indica que **no existe relación estadísticamente significativa entre el nivel de renta del entorno geográfico del cliente y el volumen de facturación que genera**. Sobre la base de 1.805 clientes nacionales con renta media documentada y actividad económica positiva, el resultado refuta la hipótesis intuitiva de que las zonas de mayor renta concentran un mayor volumen de gasto en Selmark.

Esta evidencia tiene tres implicaciones metodológicas relevantes para el resto del proyecto. En primer lugar, la variable `renta_media` no debería incorporarse como predictor principal en el clustering del Capítulo 5, dado que su capacidad discriminante sobre la facturación es prácticamente nula. En segundo lugar, las estrategias comerciales recomendadas en el Capítulo 8 no deberían basarse en una segmentación territorial por niveles de renta, sino en variables conductuales que sí presenten relación con el comportamiento de compra. En tercer lugar, este hallazgo sugiere que el cliente típico de Selmark presenta un patrón de gasto relativamente homogéneo a través de niveles socioeconómicos del entorno, lo que es coherente con la naturaleza del producto (moda íntima como bien transversal a estratos sociales).

### 4.5 Dispersión de la facturación por grupo MOSAIC

Más allá del valor medio del grupo, la distribución interna de la facturación es informativa: indica si dentro de cada grupo MOSAIC la cartera es homogénea o si presenta una elevada dispersión. Esta información será especialmente útil para el clustering del Capítulo 5, que podrá aprovechar tanto la posición central de cada grupo como su variabilidad interna.

In [33]:
# 4.5 Distribución de la facturación por grupo MOSAIC (boxplot)
box_data = df_nacional_mosaic[df_nacional_mosaic["mosaic_grupo"].notna()].copy()
box_data["facturacion_combinada"] = (
    box_data["facturacion_b2b"].fillna(0) + 
    box_data["facturacion_retail_neta_eur"].fillna(0)
)
box_data = box_data[box_data["facturacion_combinada"] > 0]

# Top 10 grupos por número de clientes (los más representativos)
top_grupos = (box_data.groupby("mosaic_grupo").size().sort_values(ascending=False).head(10).index.tolist())
box_data_top = box_data[box_data["mosaic_grupo"].isin(top_grupos)]

# Estadísticas por grupo
stats_grupos = (box_data_top.groupby("mosaic_grupo")["facturacion_combinada"]
                 .agg(["count", "median", "mean", "std"])
                 .round(0).reset_index()
                 .sort_values("median", ascending=False))

print("ESTADÍSTICAS DE FACTURACIÓN POR GRUPO MOSAIC (Top 10 grupos)")
print("=" * 90)
print(f"   {'Grupo':<10s} {'Clientes':>10s} {'Mediana':>15s} {'Media':>15s} {'Desv. Estándar':>18s}")
for _, row in stats_grupos.iterrows():
    print(f"   {str(row['mosaic_grupo']):<10s} {int(row['count']):>10,} "
          f"{fmt_es(row['median'], 0):>15s} € {fmt_es(row['mean'], 0):>15s} € {fmt_es(row['std'], 0):>16s} €")

# Boxplot con escala log
orden_grupos = stats_grupos["mosaic_grupo"].tolist()

fig = go.Figure()
for i, grupo in enumerate(orden_grupos):
    datos_g = box_data_top[box_data_top["mosaic_grupo"] == grupo]["facturacion_combinada"]
    fig.add_trace(go.Box(
        y=datos_g, name=str(grupo),
        marker_color=PALETA_CATEGORICA[i % len(PALETA_CATEGORICA)],
        boxmean=True,
        hovertemplate="<b>Grupo " + str(grupo) + "</b><br>%{y:,.0f} €<extra></extra>"
    ))
fig.update_layout(
    title=dict(text="<b>Distribución de la facturación combinada por grupo MOSAIC</b><br><sup>Top 10 grupos por número de clientes · escala logarítmica</sup>"),
    xaxis_title="Grupo MOSAIC",
    yaxis_title="Facturación combinada (€) · escala log",
    yaxis_type="log",
    height=550, showlegend=False,
)
guardar_y_mostrar(fig, "12_boxplot_facturacion_mosaic", h=550)

ESTADÍSTICAS DE FACTURACIÓN POR GRUPO MOSAIC (Top 10 grupos)
   Grupo        Clientes         Mediana           Media     Desv. Estándar
   F                  44          33.090 €          46.730 €           48.239 €
   D                  97          25.515 €          43.239 €           54.613 €
   G                 156          24.430 €          42.947 €           65.553 €
   J                  70          20.525 €          37.493 €           43.053 €
   H                 456          18.750 €          37.300 €           49.404 €
   B                 197          17.720 €          38.366 €           52.539 €
   C                 285          17.238 €          37.268 €           58.650 €
   A                 162          15.024 €          41.059 €           73.473 €
   I                 183          13.447 €          43.431 €           87.295 €
   E                 124          10.330 €          36.896 €           59.290 €


   Figura guardada en: 12_boxplot_facturacion_mosaic.png


#### Conclusiones de la dispersión MOSAIC

El análisis de dispersión por grupo MOSAIC aporta una evidencia complementaria que refuerza la conclusión anterior. Las estadísticas descriptivas de los diez grupos principales muestran un patrón consistente: las desviaciones estándar son del mismo orden de magnitud que las medias (a menudo superiores), lo que indica una heterogeneidad interna muy elevada en cada grupo. Por ejemplo, el grupo I presenta una mediana de 13.447 € pero una media de 43.431 € con una desviación estándar de 87.295 €, evidenciando la presencia de clientes con comportamientos muy dispares dentro del mismo grupo sociodemográfico.

Al comparar entre grupos, las medias se sitúan en un rango estrecho (entre 36.896 € del grupo E y 46.730 € del grupo F), mientras que las medianas presentan mayor variabilidad (entre 10.330 € y 33.090 €). El grupo F destaca como el segmento de mayor mediana pese a contar con sólo 44 clientes, mientras que los grupos más numerosos (H con 456 clientes, C con 285) presentan medianas notablemente más bajas. Esta distribución sugiere que existen subsegmentos de cliente de alto valor que no se corresponden con los grupos MOSAIC mayoritarios y que requerirán identificación mediante variables conductuales adicionales.

La conclusión metodológica de los dos análisis combinados (correlación renta-facturación nula y elevada dispersión interna en los grupos MOSAIC) es directa y constituye una de las decisiones más relevantes del proyecto: el algoritmo de clustering del Capítulo 5 deberá basarse principalmente en variables conductuales (recencia, frecuencia, mix de canales, ticket medio, ratio de devoluciones) y no en variables sociodemográficas, que se reservarán para la fase posterior de caracterización descriptiva de los segmentos obtenidos.

### Conclusiones del análisis sociodemográfico

El enriquecimiento MOSAIC sobre la cartera nacional confirma la utilidad de la fuente sociodemográfica para caracterizar el perfil del consumidor final de Selmark más allá de las variables transaccionales. La distribución por grupos MOSAIC revela la presencia de perfiles diferenciados con desigual peso en facturación, lo que sugiere que la captación de Selmark no es homogénea sobre el conjunto del territorio español sino que presenta cierta selectividad sociodemográfica.

El análisis por perfil comercial (premium, familiar joven, turístico, rural, sensible al precio) aporta una segunda lectura complementaria. La comparación entre número de clientes y facturación media por cliente identifica perfiles de alto valor unitario frente a perfiles de mayor masa con valor unitario inferior. Esta dualidad es especialmente relevante para el diseño de estrategias comerciales del Capítulo 8, donde se diferenciarán acciones orientadas a la captación masiva frente a acciones orientadas al cuidado de clientes premium de alto ticket.

El análisis de la renta media de los códigos postales con cartera nacional sitúa el perfil socioeconómico medio de los hogares donde opera Selmark, ofreciendo una referencia adicional para el cruce con los segmentos del clustering. La distribución observada permitirá identificar zonas geográficas con elevada penetración en perfiles de renta alta como objetivos prioritarios para el geomarketing del Capítulo 6.

## 5. Síntesis y hallazgos para los siguientes capítulos

In [34]:
# Resumen ejecutivo del análisis temporal y sociodemográfico
print("=" * 75)
print("SÍNTESIS DEL ANÁLISIS DESCRIPTIVO 09b")
print("=" * 75)

print(f"\n1. CARACTERIZACIÓN TEMPORAL")
print(f"   Periodo cubierto:           2022-01-01 a 2025-12-31 (1.461 días)")
print(f"   Facturación retail (sin tec): {fmt_es(evol_mensual['facturacion_neta'].sum()/1e6, 2):>8s} M €")
print(f"   Meses con datos:            {len(evol_mensual):>4d}")

print(f"\n2. EVENTOS COMERCIALES IDENTIFICADOS")
for _, row in eventos[eventos["evento"] != "Días sin evento"].iterrows():
    print(f"   {str(row['evento']):<18s}  {row['num_dias']:>4d} días  Fact/día: {fmt_es(row['facturacion_diaria_media']/1e3, 0):>5s} K €")

print(f"\n3. MIX DE CANALES (cartera sin cuentas técnicas)")
print(f"   Canales identificados:       {len(canal_dist):>4d}")
print(f"   Canal principal mayoritario: {canal_dist.iloc[0]['canal_principal']}")
total_mono = int((diversidad['num_canales_distintos'] == 1).sum())
total_multi = int((diversidad['num_canales_distintos'] >= 2).sum())
print(f"   Clientes monocanal:          {fmt_es(total_mono):>6s}  ({100*total_mono/(total_mono+total_multi):.2f} %)")
print(f"   Clientes multicanal:         {fmt_es(total_multi):>6s}  ({100*total_multi/(total_mono+total_multi):.2f} %)")

print(f"\n4. ANÁLISIS SOCIODEMOGRÁFICO MOSAIC")
print(f"   Cartera nacional con MOSAIC: {len(df_nacional_mosaic):>6,} clientes  ({100*len(df_nacional_mosaic)/int((df_sin_tecnicas['tipo_mercado']=='NACIONAL').sum()):.2f} % de la cartera nacional)")
print(f"   Grupos MOSAIC identificados: {df_nacional_mosaic['mosaic_grupo'].nunique():>6d}")
for _, row in perfiles.sort_values("num_clientes", ascending=False).iterrows():
    print(f"   Perfil {str(row['Perfil comercial']):<20s} {fmt_es(row['num_clientes']):>5s} cli ({row['pct_sobre_mosaic']:>5.2f} %)")

if len(renta_data) > 0:
    print(f"\n   Renta media de la cartera nacional: {fmt_es(renta_data['renta_media'].mean(), 0)} €/hogar")

print(f"\n" + "=" * 75)
print(f"CIERRE DEL NOTEBOOK 09b · Análisis descriptivo completo")
print(f"Figuras generadas: {len(list(RUTA_FIGURAS.glob('*.png')))} archivos PNG en {RUTA_FIGURAS.name}/")
print("=" * 75)

SÍNTESIS DEL ANÁLISIS DESCRIPTIVO 09b

1. CARACTERIZACIÓN TEMPORAL
   Periodo cubierto:           2022-01-01 a 2025-12-31 (1.461 días)
   Facturación retail (sin tec):    67,10 M €
   Meses con datos:              48

2. EVENTOS COMERCIALES IDENTIFICADOS
   Rebajas              247 días  Fact/día:    64 K €
   Navidad              124 días  Fact/día:    12 K €
   San Valentín           4 días  Fact/día:   106 K €
   Black Friday           4 días  Fact/día:    15 K €

3. MIX DE CANALES (cartera sin cuentas técnicas)
   Canales identificados:         10
   Canal principal mayoritario: Temporada
   Clientes monocanal:             792  (26.13 %)
   Clientes multicanal:          2.239  (73.87 %)

4. ANÁLISIS SOCIODEMOGRÁFICO MOSAIC
   Cartera nacional con MOSAIC:  2,036 clientes  (96.86 % de la cartera nacional)
   Grupos MOSAIC identificados:     12
   Perfil Sensible al precio     749 cli (36.79 %)
   Perfil Premium                495 cli (24.31 %)
   Perfil Familiar joven         250 cli

### Conclusiones generales del análisis temporal y sociodemográfico

El análisis temporal y sociodemográfico desarrollado en este notebook completa la caracterización descriptiva integral de la cartera de Selmark iniciada en el notebook 09a. Los hallazgos pueden agruparse en cuatro dimensiones principales que orientarán los análisis cuantitativos posteriores del trabajo.

En la **dimensión temporal**, la cartera presenta una estacionalidad muy marcada y estable a lo largo del cuatrienio, con picos sistemáticos en los periodos de rebajas (enero y julio), la campaña navideña (diciembre) y los arranques de las temporadas primavera-verano y otoño-invierno. El hallazgo más relevante de esta dimensión es la identificación de San Valentín como el evento de mayor intensidad para Selmark, con una facturación media diaria que duplica la base ordinaria. Esta evidencia, contrastada con la atipicidad del Black Friday en el sector, configura un calendario comercial claramente diferenciado del que cabría suponer en otros sectores de consumo.

En la **dimensión del mix de canales**, la cartera presenta una diversificación operativa significativa: el canal Temporada concentra el 70 % de los clientes principales y aproximadamente el 80 % de la facturación retail, mientras que el canal Repetición agrupa al 24 % de los clientes. El análisis de diversidad confirma que el 74 % de la cartera es multicanal, lo que sugiere una integración profunda del cliente típico de Selmark con la oferta de la empresa. Esta dualidad entre clientes profesionales con compromiso de temporada y clientes con dinámica de reposición a demanda constituirá una variable explicativa clave en el clustering.

En la **dimensión sociodemográfica**, el enriquecimiento MOSAIC sobre la cartera nacional (2.036 clientes con cobertura del 96,86 %) identifica una distribución de doce grupos sociodemográficos, dominada por el grupo H (24,80 %) y seguido por los grupos C, B e I. La clasificación por perfiles comerciales muestra una distribución dominada por el perfil "Sensible al precio" (36,79 %) y el perfil "Premium" (24,31 %). Resulta notable que la facturación media por cliente sea relativamente homogénea entre todos los perfiles (entre 30.000 y 44.000 euros por cliente), lo que sugiere que la diferenciación sociodemográfica no se traduce en una segmentación natural de gasto.

En la **dimensión socioeconómica territorial**, la renta media de los códigos postales con cartera nacional se sitúa en 25.891 euros por hogar como valor medio, con una mediana muy próxima (26.289 euros). El análisis de correlación entre la renta del CP y la facturación combinada del cliente aporta una primera aproximación cuantitativa a la relación entre nivel socioeconómico del entorno y comportamiento de compra, cuyo resultado será desarrollado en profundidad en el Capítulo 6.

### Hallazgos para los siguientes capítulos

- **Capítulo 5 (Clustering).** Las variables temporales (recencia, meses activo, porcentaje de pedidos de temporada y de repetición), el canal principal, los perfiles MOSAIC y la renta media del CP se incorporarán como variables candidatas para el algoritmo de clustering. La estacionalidad observada justifica el uso de la recencia como variable robusta. La dispersión interna observada en los grupos MOSAIC sugiere que el clustering podrá producir subdivisiones más finas dentro de algunos grupos.

- **Capítulo 6 (Geomarketing).** Los perfiles MOSAIC identificados (premium, familiar joven, turístico, rural y sensible al precio) se cruzarán con la distribución territorial para identificar zonas con alta penetración de cada perfil. La renta media estimada por código postal aportará una capa adicional de información para el análisis geográfico, y la correlación observada entre renta y facturación servirá como hipótesis a contrastar a escala territorial.

- **Capítulo 7 (Churn).** El análisis de cohortes por año de primera operación y la estacionalidad documentada permitirán definir umbrales de recencia ajustados por el comportamiento estacional esperado, evitando clasificar como abandonos lo que en realidad son periodos de baja actividad temporada-dependiente. El sesgo de truncamiento de la cohorte 2022 deberá tenerse en cuenta al definir la antigüedad efectiva del cliente.

- **Capítulo 8 (Recomendaciones).** Los eventos comerciales identificados, particularmente San Valentín como ventana de máximo impacto, constituirán momentos privilegiados para las acciones comerciales segmentadas que se propondrán como recomendaciones finales. Las acciones específicas para Black Friday se desaconsejarán explícitamente con base en el hallazgo cuantitativo documentado en este capítulo.

In [35]:
con.close()
print("Conexión a DuckDB cerrada. Notebook 09b completado.")
print("\nCapítulo 4 del TFG (análisis descriptivo) cerrado al 100 %.")

Conexión a DuckDB cerrada. Notebook 09b completado.

Capítulo 4 del TFG (análisis descriptivo) cerrado al 100 %.
